In [ ]:
!pip -q install streamlit supabase groq joblib scikit-learn xgboost pandas

In [ ]:
# NEW: write Streamlit theme config once (safe to re-run)
import os
os.makedirs('.streamlit', exist_ok=True)
with open('.streamlit/config.toml', 'w') as f:
    f.write('''[theme]
primaryColor              = "#5B8A72"
backgroundColor           = "#FAF7F2"
secondaryBackgroundColor  = "#F2EDE6"
textColor                 = "#1F2A2C"
font                      = "sans serif"
base                      = "light"

[server]
headless                  = true

[browser]
gatherUsageStats          = false
''')
print('Theme written to .streamlit/config.toml')

Theme written to .streamlit/config.toml


In [ ]:
import os
from google.colab import userdata
for k in ['SUPABASE_URL','SUPABASE_ANON_KEY','GROQ_API_KEY']:
      try: os.environ[k]=userdata.get(k).strip()
      except Exception as e: print('Missing secret:',k)
os.environ['MODEL_DIR']='models'
os.environ['CARECOMPASS_DEBUG']='1'   # shows technical reasons if the AI cannot be reached; set to '0' for real users

In [ ]:
# 1) Core engine: scoring, safety, plan, chat rules — no network needed
import os
import shutil
import re

# Rename files, resolving both duplicate .py extensions and browser suffixes like (4)
for filename in os.listdir('.'):
    if filename.endswith('.py') or filename.endswith('.py.py'):
        # Strip suffix
        clean_name = filename
        if clean_name.endswith('.py.py'):
            clean_name = clean_name[:-3]
        # Remove sequence suffixes like (4) or (5)
        clean_name = re.sub(r'\(\d+\)', '', clean_name)

        if clean_name != filename:
            if os.path.exists(clean_name):
                os.remove(clean_name)
            shutil.move(filename, clean_name)
            print(f"Renamed: {filename} -> {clean_name}")

import carecompass_core as cc
cc.selftest()
print('---')
print('ML models status:')
for k, v in cc.ModelService('models').status.items():
    print(f'  {k}: {v}')
print('ML model version:', cc.ModelService('models').version)

Renamed: carecompass_streamlit(8).py.py -> carecompass_streamlit.py
Renamed: carecompass_ui(1).py.py -> carecompass_ui.py
selftest passed
---
ML models status:
  Depression: loaded, but predictions stay OFF until ML_MAPPINGS_CONFIRMED=True
  Anxiety: loaded, but predictions stay OFF until ML_MAPPINGS_CONFIRMED=True
  Stress: loaded, but predictions stay OFF until ML_MAPPINGS_CONFIRMED=True
ML model version: depres:023d0341|anxiet:4ceb8746|stress:194403be


In [ ]:
# 2) Pattern mining + voice-summary helper — no network needed
import importlib
import carecompass_patterns as cpat

importlib.reload(cpat)
cpat.selftest()
print('---')
print('Demo on synthetic data:')

demo_assessments = [
    {
        'bands': {'Depression': 'Moderate', 'Anxiety': 'Mild', 'Stress': 'Severe'},
        'context': {'sleep': 'Poor', 'pressure': 'High', 'activity': 'Rarely'},
        'urgent': False
    },
    {
        'bands': {'Depression': 'Mild', 'Anxiety': 'Normal', 'Stress': 'Moderate'},
        'context': {'sleep': 'Okay', 'pressure': 'Moderate', 'activity': 'Rarely'},
        'urgent': False
    },
    {
        'bands': {'Depression': 'Normal', 'Anxiety': 'Normal', 'Stress': 'Mild'},
        'context': {'sleep': 'Good', 'pressure': 'Low', 'activity': 'A few times a week'},
        'urgent': False
    }
]

demo_checkins = [
    {'mood': 2, 'stress': 4, 'anxiety': 3, 'energy': 2, 'sleep': 2, 'feelings': ['Tired', 'Overwhelmed']},
    {'mood': 3, 'stress': 3, 'anxiety': 3, 'energy': 3, 'sleep': 3, 'feelings': ['Okay']}
]

pat = cpat.mine_patterns(demo_assessments, demo_checkins)
print('criticality:', pat['criticality'], '| score:', pat['score'])
print('reasons:', pat['reasons'])
print('trend flags:', pat['trend'])
print('rules found:', len(pat['rules']))
print('voice summary:')
print(' ', cpat.voice_summary(demo_assessments, demo_checkins, pat, 'Sam'))

carecompass_patterns selftest passed
---
Demo on synthetic data:
criticality: high | score: 12
reasons: ['Stress is severe in your most recent assessment', 'Depression has been rising across your last 3 assessments', 'Anxiety has been rising across your last 3 assessments', 'Stress has been rising across your last 3 assessments']
trend flags: ['Trend_Anxiety_Worsening', 'Trend_Depression_Worsening', 'Trend_Stress_Worsening']
rules found: 0
voice summary:
  Hi Sam. Your recent results and check-ins suggest things have been quite heavy. You do not have to manage this alone. The area asking for the most care is stress, at severe. I have noticed Stress is severe in your most recent assessment; Depression has been rising across your last 3 assessments. Please open your plan for gentle ideas that fit you, and reach out to someone you trust.


In [ ]:
import os
from google.colab import userdata
for k in ['SUPABASE_URL','SUPABASE_ANON_KEY','GROQ_API_KEY']:
      try: os.environ[k]=userdata.get(k).strip()
      except Exception as e: print('Missing secret:',k)
os.environ['MODEL_DIR']='models'
os.environ['CARECOMPASS_DEBUG']='1'   # shows technical reasons if the AI cannot be reached; set to '0' for real users

In [ ]:
# Groq connection test: run this if the chat or the plan ever falls back to a standard reply
import carecompass_core as cc

print('Models tried, in order:', cc.GROQ_MODELS)
t = cc.groq_complete([{'role': 'user', 'content': 'Reply with one friendly sentence.'}], temperature=0.2, max_tokens=400)
print('Groq says:', t)
print('Problem  :', cc.LAST_GROQ_ERROR or 'none')

Models tried, in order: ['openai/gpt-oss-120b', 'openai/gpt-oss-20b', 'qwen/qwen3.6-27b']
Groq says: Sure thing—hope your day is off to a wonderful start!
Problem  : none


In [ ]:
!pkill -f streamlit
!pkill -f cloudflared

In [ ]:
import os
import subprocess
import time
import re
import urllib.request

# stop any old copy still running with old code/keys
subprocess.run('pkill -f streamlit; pkill -f cloudflared', shell=True)
time.sleep(2)

# start streamlit in the background
subprocess.Popen('streamlit run carecompass_streamlit.py --server.port 8501 --server.headless true > streamlit.log 2>&1', shell=True)
time.sleep(6)

# download cloudflared tunnel client
urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', 'cloudflared')
os.chmod('cloudflared', 0o755)

# establish the tunnel
p = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8501'], stderr=subprocess.PIPE, text=True)

# read stderr line by line until the trycloudflare URL is generated
for line in p.stderr:
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m:
        print('OPEN THIS LINK:', m.group(0))
        break

# keep this cell running while you use the app; if the page errors, run: !tail -30 streamlit.log

OPEN THIS LINK: https://fotos-students-moore-note.trycloudflare.com
